# Financial Data Analysis & Visualization

This notebook processes accounting data (imported from Excel/CSV), computes financial metrics, and visualizes trends.
It is deprecated in favour of the GUI, but uses the same filtering model.

**Key Phases:**
1.  **Data Loading**: Scans `../data/etica` for `.xls` files, converts and merges them (`load_data`), then gives every movement a `Class` (`cluster_merged`, from the rules in `custom-classes.csv`; unmatched movements are `todo`).
2.  **Views**: Filters the data into views. A view is the tuple of classes it excludes, the same as unchecking class checkboxes in the GUI.
3.  **Processing**: Each view gets its own running total and OLS linear regression, then candlesticks.
4.  **Visualization**: Static plots (Matplotlib) and interactive plots (Plotly).

In [ ]:
from piggy_bank.etica import *

# Ensure plots display in the notebook
%matplotlib inline

init_logging(also_time=False)

print('Loading data from', DATA_DIR)

## 1. Data Loading & Preprocessing

In [ ]:
_df_all = cluster_merged(load_data())

print(f"Total rows in processed data: {_df_all.shape[0]}")
print(_df_all[COL_CLASS].value_counts())
print(_df_all.head())

---
## 2. Views

Edit `DEFAULT_VIEWS` (or loop over your own tuples) to compare any combination, e.g. `('blacklist', 'investment')` is the data without those two classes.

* **`build_view`**: Filters with `apply_classes`, then computes the view's own running total (`compute_running_total`) and OLS fit (`compute_ols`, slope `m` in EUR/day and intercept `q`).
* **`compute_candlesticks`**: Groups a view's running total into OHLC candlesticks per week (from Monday), month (from the 1st) or year.

In [ ]:
_views = {}
for _exclude in DEFAULT_VIEWS:
    _df_view, _m, _q = build_view(_df_all, exclude=_exclude)
    _views[view_label(_exclude)] = _df_view
    print_view_summary(_df_view, view_label(_exclude), _m)

## Main Plot: Running Total and OLS

In [ ]:
plot_running_total_ols(_views)

In [ ]:
plot_running_total_ols_interactive(_views)

---
# Candlesticks

Also the derivative of the closing balance (Change in `Close`).

In [ ]:
# Compute. TF_WEEKLY (weeks start on Monday), TF_MONTHLY (on the 1st) or TF_YEARLY
_cndl_lbl = TF_MONTHLY
_candles = {lbl: compute_candlesticks(df, _cndl_lbl) for lbl, df in _views.items()}

In [ ]:
for _lbl, _df_candles in _candles.items():
    plot_candlestick(_df_candles, f'{_cndl_lbl} {_lbl}')

In [ ]:
for _lbl, _df_candles in _candles.items():
    plot_candlestick_interactive(_df_candles, _lbl)

---
## 3. Single View Detail

The running total of one view as bars, e.g. the last of `DEFAULT_VIEWS`.

In [ ]:
_lbl = view_label(DEFAULT_VIEWS[-1])
plot_view_bars(_views[_lbl], _lbl)